# How Much of a Transformer Is Code? — 4B substitution

Measures the **thesis-critical number the paper is missing at scale**: how much loss it costs to replace ~36% of attention heads (positional templates) + ~21% of MLP layers (token lookup tables) in **Qwen3-4B**, after healing only the normalization gains. The scientific comparison is a paired run of this exact protocol on 0.6B and 4B; do not compare the WikiText protocol directly to the paper's fiction-domain +0.88 figure.

**Before running:** `Runtime → Change runtime type → GPU` (an **L4 or A100 / 24 GB+** — the free T4 will likely OOM on the 4B heal).

Then `Runtime → Run all`. It auto-pulls WikiText-103 (no upload) and prints the headline block.

Repo: https://github.com/keithagroves/how-much-transformer-is-code

In [ ]:
!pip install -q transformers datasets accelerate
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU — set Runtime to GPU')

In [ ]:
#@title Optional · Cache models in Google Drive (survives runtime restarts)
# Without this, a new Colab VM re-downloads the model (~8 GB for 4B).
# With it, the HuggingFace cache lives in your Drive (needs ~2–8 GB free per model).
USE_DRIVE_CACHE = False  #@param {type:"boolean"}
if USE_DRIVE_CACHE:
    from google.colab import drive; drive.mount('/content/drive')
    import os; os.makedirs('/content/drive/MyDrive/hf_cache', exist_ok=True)
    %env HF_HOME=/content/drive/MyDrive/hf_cache
    print('HF cache -> Google Drive')
else:
    print('using the VM-local cache (re-downloads on a fresh runtime)')


In [ ]:
#@title Settings { run: "auto" }
MODEL = "Qwen/Qwen3-4B"  #@param ["Qwen/Qwen3-4B", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-0.6B"]
HEAD_FRACTION = 0.36  #@param {type:"slider", min:0.1, max:0.5, step:0.02}
HEAL_EPOCHS = 12  #@param {type:"slider", min:4, max:20, step:1}
print(MODEL, HEAD_FRACTION, HEAL_EPOCHS)

In [ ]:
#@title Step 1 · Download the model (instant if already cached)
from huggingface_hub import snapshot_download
path = snapshot_download(MODEL)
print('model ready:', path)


In [ ]:
#@title Step 2 · Run the experiment (re-run freely — reuses the downloaded model)
# pull the latest pipeline script from the repo and run it
!wget -q -O sub.py https://raw.githubusercontent.com/keithagroves/how-much-transformer-is-code/main/src/colab_4b_substitution.py
!python sub.py "$MODEL" "$HEAD_FRACTION" "$HEAL_EPOCHS"


### Reading the output
- First run **Qwen3-0.6B**, then **Qwen3-4B** with the same head fraction and epoch cap. Compare **code + heal** and **fair cost** between those paired runs.
- **zero-ablation + heal** is the control — code should sit well below it, or the substitution isn't carrying function.
- If 4B's fair cost is no larger than 0.6B's under this matched protocol, substitutability holds or improves with scale; a materially larger 4B cost means scale shrinks the replaceable fraction.

Note: this uses **positional-only templates** (content-free), justified because the paper's positional-vs-fitted sweep shows content columns buy ~nothing in the substitutable set.